# Recurrent Neural Networks (RNNs) - Solutions

Worked solutions for the exercises. The lesson code is kept because the exercises build on it, so this notebook runs end to end on its own.

## Setup and Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping

RSEED = 42
np.random.seed(RSEED)
tf.random.set_seed(RSEED)

print(f"TensorFlow Version: {tf.__version__}")

In [ ]:
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["figure.titlesize"] = 14
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["figure.figsize"] = (10, 6)

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"GPU available: {len(gpus)} GPU(s)")
    for gpu in gpus:
        print(f"  - {gpu}")
else:
    print("No GPU found, using CPU")

## Part 1: Simple RNN for Sine Prediction

We start with a simple example: predicting future values of a sine function. This is an ideal task for understanding how RNNs work.

### Generate Data

In [ ]:
def generate_sine_wave(seq_length=100, num_sequences=1000):
    # Generate random sine wave sequences
    X = []
    y = []

    for _ in range(num_sequences):
        start = np.random.uniform(0, 10)
        freq = np.random.uniform(0.5, 2.0)

        t = np.linspace(start, start + 4 * np.pi, seq_length + 1)
        sine_wave = np.sin(freq * t)

        X.append(sine_wave[:-1])
        y.append(sine_wave[-1])

    X = np.array(X).reshape(-1, seq_length, 1).astype("float32")
    y = np.array(y).reshape(-1, 1).astype("float32")

    return X, y


X_train, y_train = generate_sine_wave(seq_length=50, num_sequences=800)
X_test, y_test = generate_sine_wave(seq_length=50, num_sequences=200)

print(f"Training data shape: X={X_train.shape}, y={y_train.shape}")
print(f"Test data shape: X={X_test.shape}, y={y_test.shape}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

for i in range(4):
    axes[i].plot(X_train[i], label="Input sequence")
    axes[i].axhline(
        y=y_train[i], color="r", linestyle="--", label=f"Target: {y_train[i][0]:.3f}"
    )
    axes[i].set_xlabel("Time step")
    axes[i].set_ylabel("Value")
    axes[i].set_title(f"Example sequence {i + 1}")
    axes[i].legend()
    axes[i].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Implementation of a Simple RNN

A `SimpleRNN` layer applies the recurrence above at every step and returns the hidden state. We set `return_sequences=False` so it returns only the final state, which is what a single-value prediction needs. Setting it to `True` returns the state at every step, which you want when stacking recurrent layers or predicting one value per step.

A `Dense` layer then maps that final state to a single number.

The alternative would be to flatten the whole sequence and feed it to a plain `Dense` network. That works, but it treats each position as an independent feature: it learns nothing about order, and a model trained on 50 steps cannot accept 60. The recurrence is what makes the model reusable across lengths.

In [ ]:
def build_simple_rnn(input_shape, hidden_size=32, output_size=1):

    model = models.Sequential(
        [
            layers.Input(shape=input_shape),
            layers.SimpleRNN(hidden_size, return_sequences=False),
            layers.Dense(output_size),
        ]
    )

    return model


model_rnn = build_simple_rnn(input_shape=(50, 1), hidden_size=32, output_size=1)

model_rnn.compile(
    optimizer=optimizers.Adam(learning_rate=0.01), loss="mse", metrics=["mae"]
)

print(model_rnn.summary())
print(f"\nNumber of parameters: {model_rnn.count_params()}")

### Training the RNN

In [ ]:
history_rnn = model_rnn.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1,
)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history_rnn.history["loss"], label="Training loss", alpha=0.7)
plt.plot(history_rnn.history["val_loss"], label="Validation loss", alpha=0.7)
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.title("Training and validation loss over epochs (Simple RNN)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
predictions = model_rnn.predict(X_test)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

for i in range(4):
    idx = i * 10
    axes[i].plot(X_test[idx], label="Input sequence", alpha=0.7)
    axes[i].axhline(
        y=y_test[idx], color="g", linestyle="--", label=f"Actual: {y_test[idx][0]:.3f}"
    )
    axes[i].axhline(
        y=predictions[idx],
        color="r",
        linestyle="--",
        label=f"Prediction: {predictions[idx][0]:.3f}",
    )
    axes[i].set_xlabel("Time step")
    axes[i].set_ylabel("Value")
    axes[i].set_title(f"Prediction example {i + 1}")
    axes[i].legend()
    axes[i].grid(alpha=0.3)

plt.tight_layout()
plt.show()

mae = np.mean(np.abs(predictions - y_test))
print(f"\nMean Absolute Error: {mae:.4f}")

## Part 2: The Vanishing Gradient Problem

Training an RNN uses backpropagation through time. To update the weights from a loss computed at the last step, the gradient has to travel back through every earlier step, and each step back multiplies it by the same recurrent Jacobian. That Jacobian is dominated by the recurrent weight matrix $W_{hh}$.

After $t$ steps the gradient has been multiplied by $W_{hh}$ about $t$ times, so its size is governed by the largest absolute eigenvalue of $W_{hh}$, known as the spectral radius $\rho$:

* $\rho < 1$: the gradient shrinks geometrically, which is the vanishing gradient problem
* $\rho > 1$: the gradient grows geometrically, which is the exploding gradient problem
* $\rho = 1$: a knife-edge case that preserves the signal but that no optimiser can hold exactly

We can measure this decay directly rather than inferring it from a training run. Whether a model trains well depends on initialisation, learning rate and luck, but this matrix product does not: it is the same every time you run it.

In [ ]:
# Measure how much of the gradient survives after t steps back through time.
# The same random matrix is rescaled to each spectral radius, so the radius is
# the only thing that differs between the curves.


def gradient_factor_over_time(spectral_radius, steps=100, hidden_size=32, seed=RSEED):
    rng = np.random.default_rng(seed)
    W = rng.standard_normal((hidden_size, hidden_size))
    W = W / max(abs(np.linalg.eigvals(W))) * spectral_radius

    product = np.eye(hidden_size)
    factors = []
    for _ in range(steps):
        product = product @ W
        factors.append(np.linalg.norm(product))

    return np.array(factors)


radii = [0.8, 0.95, 1.0, 1.05]
decay = {r: gradient_factor_over_time(r) for r in radii}

for r, factors in decay.items():
    print(f"Spectral radius {r}: gradient factor after 100 steps = {factors[-1]:.3e}")

In [ ]:
plt.figure(figsize=(10, 6))

for r, factors in decay.items():
    plt.plot(
        range(1, len(factors) + 1), factors, label=f"Spectral radius {r}", linewidth=2
    )

plt.axhline(y=1.0, color="grey", linestyle="--", alpha=0.6)
plt.yscale("log")
plt.xlabel("Steps back through time")
plt.ylabel("Gradient factor (log scale)")
plt.title("How the gradient scales as it travels back through a simple RNN")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Part 3: How an LSTM Preserves the Gradient

An LSTM carries a separate cell state $c_t$ alongside the hidden state, and updates it additively under the control of a forget gate $f_t$:

$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$

Along that cell-state path the derivative from one step to the next is just $f_t$, so the factor over 100 steps is the product of the forget gates rather than a product of weight matrices.

That difference is the whole idea. In a simple RNN the shrinkage is fixed by a weight matrix that also has to do the learning, so it cannot be tuned independently. In an LSTM the network can learn to hold the gate open for information worth keeping, and close it for information worth discarding, per step and per feature.

```mermaid
flowchart LR
    cprev[Cell state at t-1] --> keep[Multiply by forget gate]
    keep --> combine[Add new candidate]
    combine --> cnow[Cell state at t]
    forget[Forget gate] --> keep
    inp[Input gate] --> combine
    cand[Candidate value] --> combine
```

The path along the top of that diagram is the one that matters for gradients: it is addition and scaling, with no weight matrix in the way.

In [ ]:
# The gradient factor along the LSTM cell state is the product of the forget gates.

forget_gates = [0.5, 0.9, 0.99, 1.0]
steps = 100

for f in forget_gates:
    print(
        f"Forget gate held at {f}: gradient factor after {steps} steps = {f**steps:.3e}"
    )

In [ ]:
timesteps = np.arange(1, 101)

plt.figure(figsize=(10, 6))

for f in forget_gates:
    plt.plot(timesteps, f**timesteps, label=f"LSTM, forget gate {f}", linewidth=2)

plt.plot(
    timesteps,
    decay[0.8],
    label="Simple RNN, spectral radius 0.8",
    linestyle="--",
    color="black",
    linewidth=2,
)

plt.axhline(y=1.0, color="grey", linestyle="--", alpha=0.6)
plt.yscale("log")
plt.xlabel("Steps back through time")
plt.ylabel("Gradient factor (log scale)")
plt.title("Gradient flow: LSTM cell state versus a simple RNN")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
def build_lstm_model(input_shape, hidden_size=32, num_layers=1, output_size=1):

    model = models.Sequential()

    model.add(layers.Input(shape=input_shape))
    model.add(layers.LSTM(hidden_size, return_sequences=(num_layers > 1)))

    for i in range(1, num_layers):
        return_seq = i < num_layers - 1
        model.add(layers.LSTM(hidden_size, return_sequences=return_seq))

    model.add(layers.Dense(output_size))

    return model


model_lstm = build_lstm_model(
    input_shape=(100, 1), hidden_size=32, num_layers=1, output_size=1
)
model_lstm.compile(
    optimizer=optimizers.Adam(learning_rate=0.01), loss="mse", metrics=["mae"]
)

print(model_lstm.summary())

## Part 4: GRU - Alternative to LSTM

In [ ]:
def build_gru_model(input_shape, hidden_size=32, num_layers=1, output_size=1):

    model = models.Sequential()

    model.add(layers.Input(shape=input_shape))
    model.add(layers.GRU(hidden_size, return_sequences=(num_layers > 1)))

    for i in range(1, num_layers):
        return_seq = i < num_layers - 1
        model.add(layers.GRU(hidden_size, return_sequences=return_seq))

    model.add(layers.Dense(output_size))

    return model


model_gru = build_gru_model(
    input_shape=(100, 1), hidden_size=32, num_layers=1, output_size=1
)
model_gru.compile(
    optimizer=optimizers.Adam(learning_rate=0.01), loss="mse", metrics=["mae"]
)

print(model_gru.summary())

In [ ]:
# Gating is not free. Compare the parameter counts at the same hidden size.

print("Parameters at hidden size 32, one output unit:")
print(f"Simple RNN: {model_rnn.count_params():>6}")
print(f"GRU:        {model_gru.count_params():>6}")
print(f"LSTM:       {model_lstm.count_params():>6}")

## Part 5: Practical Application - Time Series Forecasting

Now we apply our knowledge to a more realistic task: forecasting passenger numbers in a time series.

The series below combines three ingredients: a slow upward trend, a seasonal cycle that repeats every 50 steps, and random noise. We normalise it before training, because the raw values sit around 100 and inputs that large push a recurrent layer's activations into saturation, where the gradients are small.

We give the model a window of 20 consecutive points and ask for the next one. An alternative is to predict several steps ahead at once, which avoids feeding predictions back into the model but needs more output units and more data to fit them.

In [ ]:
def create_sequences(data, seq_length):
    # Create sequences for time series
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i : i + seq_length])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)


np.random.seed(RSEED)
t = np.arange(500)
trend = 0.02 * t
seasonal = 10 * np.sin(2 * np.pi * t / 50)
noise = np.random.randn(500) * 2
passengers = 100 + trend + seasonal + noise

mean_pass = passengers.mean()
std_pass = passengers.std()
passengers_norm = (passengers - mean_pass) / std_pass

seq_length = 20
X_ts, y_ts = create_sequences(passengers_norm, seq_length)
X_ts = X_ts.reshape(-1, seq_length, 1).astype("float32")
y_ts = y_ts.reshape(-1, 1).astype("float32")

split = int(0.8 * len(X_ts))
X_train_ts, X_test_ts = X_ts[:split], X_ts[split:]
y_train_ts, y_test_ts = y_ts[:split], y_ts[split:]

print(f"Time series data: {X_train_ts.shape} / {X_test_ts.shape}")

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(t, passengers, label="Passengers", alpha=0.7)
plt.axvline(x=split + seq_length, color="r", linestyle="--", label="Train/test split")
plt.xlabel("Time")
plt.ylabel("Number of passengers")
plt.title("Passenger time series with trend and seasonality")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
model_ts = build_lstm_model(
    input_shape=(20, 1), hidden_size=64, num_layers=2, output_size=1
)
model_ts.compile(
    optimizer=optimizers.Adam(learning_rate=0.001), loss="mse", metrics=["mae"]
)

print("Training LSTM for time series forecasting:")
history_ts = model_ts.fit(
    X_train_ts,
    y_train_ts,
    epochs=100,
    batch_size=32,
    validation_data=(X_test_ts, y_test_ts),
    verbose=1,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
    ],
)

In [ ]:
predictions_ts = model_ts.predict(X_test_ts)
y_test_denorm = y_test_ts * std_pass + mean_pass
predictions_denorm = predictions_ts * std_pass + mean_pass

test_indices = np.arange(split + seq_length, len(passengers))

plt.figure(figsize=(14, 6))
plt.plot(t, passengers, label="Actual data", alpha=0.7)
plt.plot(
    test_indices,
    predictions_denorm,
    label="LSTM predictions",
    color="r",
    alpha=0.7,
    linewidth=2,
)
plt.axvline(
    x=split + seq_length, color="g", linestyle="--", label="Test start", alpha=0.5
)
plt.xlabel("Time")
plt.ylabel("Number of passengers")
plt.title("LSTM time series forecasting")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

mae_ts = np.mean(np.abs(predictions_denorm - y_test_denorm))
rmse_ts = np.sqrt(np.mean((predictions_denorm - y_test_denorm) ** 2))

print(f"\nMean Absolute Error: {mae_ts:.2f}")
print(f"Root Mean Squared Error: {rmse_ts:.2f}")

## Exercise Solutions

### Exercise 1: Bidirectional LSTM

Train a bidirectional LSTM on the time series data and compare its performance with the unidirectional LSTM.

In [ ]:
# A Bidirectional wrapper runs one copy of the layer forward over the window and
# another backward, then concatenates both outputs, so the layer output doubles in width.

model_bi = models.Sequential(
    [
        layers.Input(shape=(seq_length, 1)),
        layers.Bidirectional(layers.LSTM(64, return_sequences=True)),
        layers.Bidirectional(layers.LSTM(64)),
        layers.Dense(1),
    ]
)

model_bi.compile(
    optimizer=optimizers.Adam(learning_rate=0.001), loss="mse", metrics=["mae"]
)

history_bi = model_bi.fit(
    X_train_ts,
    y_train_ts,
    epochs=100,
    batch_size=32,
    validation_data=(X_test_ts, y_test_ts),
    verbose=0,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
    ],
)

predictions_bi = model_bi.predict(X_test_ts, verbose=0)
predictions_bi_denorm = predictions_bi * std_pass + mean_pass
mae_bi = np.mean(np.abs(predictions_bi_denorm - y_test_denorm))

print(f"Unidirectional LSTM: MAE {mae_ts:.2f}, {model_ts.count_params()} parameters")
print(f"Bidirectional LSTM:  MAE {mae_bi:.2f}, {model_bi.count_params()} parameters")

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(history_ts.history["val_loss"], label="Unidirectional", linewidth=2)
plt.plot(history_bi.history["val_loss"], label="Bidirectional", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Validation MSE loss")
plt.title("Unidirectional versus bidirectional LSTM")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Explanation**

A bidirectional layer reads the input window forwards and backwards, then concatenates both hidden states, so it roughly doubles the parameters. That is legitimate here because the whole window is already history relative to the target, but it would not be for streaming forecasts where later values do not exist yet. Compare the two printed MAE values: on a smooth trend plus season the reverse pass adds little, because the most informative points are the ones nearest the target and the forward pass already sees them.


### Exercise 2: Hyperparameters

Experiment with different values for `hidden_size` and `num_layers`. How does this affect performance and training time?

In [ ]:
# Sweep a small grid. Each configuration is trained with the same callback and data,
# so differences come from capacity alone rather than from the training budget.

import time

configs = [
    {"hidden_size": 16, "num_layers": 1},
    {"hidden_size": 64, "num_layers": 1},
    {"hidden_size": 16, "num_layers": 2},
    {"hidden_size": 64, "num_layers": 2},
]

results = []

for cfg in configs:
    model_cfg = build_lstm_model(input_shape=(seq_length, 1), output_size=1, **cfg)
    model_cfg.compile(
        optimizer=optimizers.Adam(learning_rate=0.001), loss="mse", metrics=["mae"]
    )

    start = time.time()
    history_cfg = model_cfg.fit(
        X_train_ts,
        y_train_ts,
        epochs=60,
        batch_size=32,
        validation_data=(X_test_ts, y_test_ts),
        verbose=0,
        callbacks=[
            EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
        ],
    )
    elapsed = time.time() - start

    predictions_cfg = model_cfg.predict(X_test_ts, verbose=0) * std_pass + mean_pass
    results.append(
        {
            **cfg,
            "params": model_cfg.count_params(),
            "epochs_run": len(history_cfg.history["loss"]),
            "seconds": round(elapsed, 1),
            "mae": round(float(np.mean(np.abs(predictions_cfg - y_test_denorm))), 3),
        }
    )

print(
    f"{'units':>6} {'layers':>7} {'params':>8} {'epochs':>7} {'seconds':>8} {'MAE':>7}"
)
for r in results:
    print(
        f"{r['hidden_size']:>6} {r['num_layers']:>7} {r['params']:>8} "
        f"{r['epochs_run']:>7} {r['seconds']:>8} {r['mae']:>7}"
    )

**Explanation**

Capacity is swept along two axes at once, width (hidden_size) and depth (num_layers), with early stopping so each configuration trains as long as it needs rather than for a fixed budget. Read the table by columns: parameters and seconds grow quickly with both axes, while MAE flattens once the model is large enough to represent a trend plus a 50 step cycle. The smallest configuration that reaches the error floor is the one to prefer, since the extra capacity buys training time rather than accuracy.


### Exercise 3: Sequence Length

What happens when you change the sequence length in time series forecasting (for example to 5, 10, 50)? Is there an optimal value?

In [ ]:
# Rebuild the windows at each length. The series is unchanged: only how much history
# the model sees per prediction differs, so this isolates the effect of the window.

window_results = []

for window in [5, 10, 20, 50]:
    X_w, y_w = create_sequences(passengers_norm, window)
    X_w = X_w.reshape(-1, window, 1).astype("float32")
    y_w = y_w.reshape(-1, 1).astype("float32")

    split_w = int(0.8 * len(X_w))
    X_train_w, X_test_w = X_w[:split_w], X_w[split_w:]
    y_train_w, y_test_w = y_w[:split_w], y_w[split_w:]

    model_w = build_lstm_model(
        input_shape=(window, 1), hidden_size=64, num_layers=2, output_size=1
    )
    model_w.compile(
        optimizer=optimizers.Adam(learning_rate=0.001), loss="mse", metrics=["mae"]
    )

    model_w.fit(
        X_train_w,
        y_train_w,
        epochs=60,
        batch_size=32,
        validation_data=(X_test_w, y_test_w),
        verbose=0,
        callbacks=[
            EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
        ],
    )

    predictions_w = model_w.predict(X_test_w, verbose=0) * std_pass + mean_pass
    actual_w = y_test_w * std_pass + mean_pass
    window_results.append((window, float(np.mean(np.abs(predictions_w - actual_w)))))

print(f"{'window':>7} {'MAE':>8}")
for window, mae_w in window_results:
    print(f"{window:>7} {mae_w:>8.3f}")

plt.figure(figsize=(9, 5))
plt.plot(
    [w for w, _ in window_results],
    [m for _, m in window_results],
    marker="o",
    linewidth=2,
)
plt.xlabel("Input window length (time steps)")
plt.ylabel("Test MAE (passengers)")
plt.title("Forecast error against input window length")
plt.grid(alpha=0.3)
plt.show()

**Explanation**

Each window length rebuilds the dataset, so this isolates how much history the model sees per prediction. Very short windows cannot contain a full seasonal cycle of 50 steps, so the model has to infer the phase from too little evidence. Very long windows contain the cycle comfortably but give the model more noise to overfit and fewer training examples, since each window consumes more of the series. The best value is usually the shortest window that still spans one full cycle.


### Exercise 4: Real-World Application

Download a real time series dataset (for example stock prices or temperature) and apply the techniques you learned.

This exercise is deliberately left open: the point is to meet the problems that synthetic data hides, such as missing timestamps, irregular sampling, outliers and regime changes. Reuse `create_sequences` and `build_lstm_model` unchanged, and keep three habits from this notebook. Normalise using statistics computed on the training split only, never on the whole series, or you leak future information into the scaling. Split chronologically rather than randomly, since shuffling lets the model see the future. Always compare against a naive baseline such as predicting the previous value, because on many real series that baseline is hard to beat.